# Milestone 5: where the sample goes wrong

This notebook is generated by `tools/build_notebooks.py`. Edit the builder, not this file: a test fails if the two drift apart.

It takes the course's mistakes one at a time and prints what each one costs on AAPL. Most numbers come from the committed files in `app/public/data/`; two cells refit the course's own ARIMA(0, 1, 1) on the committed training rows (a second or two each). The two crashes are reproduced on made-up numbers. The walkthrough is `docs/milestones/05-where-the-sample-goes-wrong.md`.

In [ ]:
import math

import numpy as np
from scipy import stats

from arima_forecasting.artifacts import find_app_data, read_artifact, table_from_prices
from arima_forecasting.forecasting import windows

app_data = find_app_data()
ev = read_artifact(app_data, "evaluation")["data"]
fit = read_artifact(app_data, "fit")["data"]
course_static = read_artifact(app_data, "forecast_course_static")["data"]
metrics = {(r["model_id"], r["method"], r["window"]): r for r in ev["metrics"]}
random_walk = {(r["kind"], r["window"]): r for r in ev["random_walk"]}

## 1. The order is chosen on the whole series

The course runs `auto_arima` on 2015 to 2023, test period included, and only then splits. `fit.json` records the same search on the course window and on the training rows alone.

In [ ]:
track = fit["course_track"]
print(f"course window: {track['n_course']} rows ({track['test_rows_seen']} of them test rows): order {track['full_series']['order']}, trend {track['full_series']['trend']!r}")
print(f"training rows: {track['n_train']} rows: order {track['train_only']['order']}, trend {track['train_only']['trend']!r}; same order: {track['same_order']}")

## 2. The static path is called the forecast

The course forecasts all 227 test days from the last training day in one call. For an ARIMA(0, 1, 1) without a drift every step after the first is the same number: a flat line.

In [ ]:
for w in course_static["windows"]:
    median = [r["median_usd"] for r in w["rows"]]
    actual = [r["actual_usd"] for r in w["rows"]]
    row = metrics[("course", "static", w["id"])]
    base = random_walk[("multi_step", w["id"])]
    print(f"{w['id']:>4}: the median path runs ${min(median):.2f} to ${max(median):.2f}; the price ${min(actual):.2f} to ${max(actual):.2f}. RMSE ${row['usd_median']['rmse']:.3f} against the flat line's ${base['usd']['rmse']:.3f} (ratio {row['ratio_to_rw']['usd_median']['rmse']:.4f})")

## 3. A dollar error with no baseline

Beside each one-step metric, the random walk's: yesterday's price. A ratio above 1 means yesterday's price was closer. The DM verdicts are DM-HLN, per test, uncorrected for the 16 tests.

In [ ]:
kinds = {m["id"]: m["kind"] for m in ev["methods"]}
for row in ev["metrics"]:
    if kinds[row["method"]] == "one_step":
        print(f"{row['model_id']:>6} {row['method']:>12} {row['window']:>4}: RMSE ratio {row['ratio_to_rw']['usd_median']['rmse']:.4f}")
for verdict in ("beats_rw", "no_difference", "worse_than_rw"):
    print(f"DM-HLN verdict {verdict}: {sum(r['verdict'] == verdict for r in ev['dm'])} of {len(ev['dm'])} tests")

## 4. exp of a log forecast is the median price, not the mean

The model forecasts the log price as a normal with mean mu and variance s^2. exp(mu) is the median of the price; its mean is exp(mu + s^2 / 2).

In [ ]:
for w in course_static["windows"]:
    last = w["rows"][-1]
    print(f"{w['id']:>4}, last day {last['date']}: median ${last['median_usd']:.2f}, mean ${last['mean_usd']:.2f} ({100 * (last['mean_usd'] / last['median_usd'] - 1):.2f}% higher), actual ${last['actual_usd']:.2f}")

## 5. The refit drops the drift

The search chose (0, 1, 1) with a constant, which at d = 1 is a drift. The course refits with `ARIMA(train, order=(0, 1, 1)).fit()`, and statsmodels' ARIMA has no trend at d >= 1 unless asked. This cell refits the same order with a drift, as `fitting.py` does, and forecasts it statically.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

from arima_forecasting.fitting import COV_TYPE
from arima_forecasting.forecasting import Spec, static_forecast
from arima_forecasting.selection import MAXITER, OPTIMIZER

table = table_from_prices(read_artifact(app_data, "prices")["data"])
y = table["log_close"].to_numpy()
test, oos = windows(table)
drifted = ARIMA(y[: test.start], order=(0, 1, 1), trend="t", enforce_stationarity=True, enforce_invertibility=True, concentrate_scale=False).fit(
    method="statespace", method_kwargs={"method": OPTIMIZER, "maxiter": MAXITER}, cov_type=COV_TYPE
)
print("drift per day:", round(float(drifted.params[0]), 7), "- fit.json says", round(track["drift_dropped"]["drift"], 7))
spec = Spec((0, 1, 1), "t", np.asarray(drifted.params, dtype=float))
for w in (test, oos):
    mean, _ = static_forecast(y, w.start, w.stop - w.start, spec)
    actual = np.exp(y[w.start : w.stop])
    with_drift = float(np.sqrt(np.mean((actual - np.exp(mean)) ** 2)))
    print(f"{w.id:>4}: static RMSE with the drift ${with_drift:.3f}, without (the course's) ${metrics[('course', 'static', w.id)]['usd_median']['rmse']:.3f}")

## 6. The QQ plot keeps the first residual

With d = 1, statsmodels starts the filter from a diffuse state, so the first one-step forecast knows nothing and the first residual is the first log price itself. `diagnostics.burn_in` drops it, by statsmodels' own rule.

In [ ]:
from arima_forecasting.fitting import course_refit

result, _ = course_refit(y[: test.start], (0, 1, 1))
resid = np.asarray(result.resid, dtype=float)
top = np.sort(resid)[::-1]
print(f"first residual {resid[0]:.4f}, first log price {y[0]:.4f}; largest two residuals {top[0]:.4f} and {top[1]:.4f}")
for label, x in (("with the first residual", resid), ("without it", resid[1:])):
    print(f"{label:>24}: kurtosis {stats.kurtosis(x, fisher=False):.2f}")

## 7. Two lines of the course that no longer run

Module 10's MAPE: `yfinance` 1.7.0 returns the close as a one-column DataFrame. Subtracting it from a Series lines the Series' index up against the column labels, so every cell is NaN. Made-up prices:

In [ ]:
import pandas as pd

dates = pd.to_datetime(["2023-12-27", "2023-12-28", "2023-12-29"])
actual = pd.DataFrame({"AAPL": [100.0, 102.0, 101.0]}, index=dates)
forecast = pd.Series([99.0, 103.0, 100.0], index=dates)
print("the course's MAPE:", float(np.mean(np.abs((actual - forecast) / actual)) * 100))
print("on one-dimensional arrays:", round(float(np.mean(np.abs((actual["AAPL"].to_numpy() - forecast.to_numpy()) / actual["AAPL"].to_numpy())) * 100), 4))

Module 9's forecast: a model fitted to a Series indexed by trading dates, which have no fixed frequency (holidays), cannot forecast past its sample in statsmodels 0.15. A simulated walk with three days taken out:

In [ ]:
import warnings

days = pd.bdate_range("2021-01-04", periods=63).delete([10, 30, 50])
walk = pd.Series(np.cumsum(np.random.default_rng(5).normal(0, 0.01, len(days))), index=days)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        ARIMA(walk, order=(0, 1, 1)).fit().get_forecast(steps=5)
        print("forecast ran")
    except ValueError as error:
        print("ValueError:", error)